# UN Speeches — Preprocessing + Topic Classification

Rebuilds `speeches_paragraphs.csv` from the raw `all_speeches.csv`.

**Runtime**: CPU is fine for preprocessing. Use GPU to speed up topic classification.

**Upload this file before running Cell 2:**
- `all_speeches.csv`
- `topic_definitions.csv` (from `raw_data/`)
- `clean_countries.py` (from `streamlit/`)

In [ ]:
# ── Cell 1: Install dependencies ───────────────────────────────────────────────
!pip install -q sentence-transformers umap-learn scikit-learn pandas

In [ ]:
# ── Cell 2: Upload files ───────────────────────────────────────────────────────
from google.colab import files
print('Upload: all_speeches.csv, topic_definitions.csv, clean_countries.py')
uploaded = files.upload()

In [ ]:
# ── Cell 3: Load and harmonize raw speeches ────────────────────────────────────
import pandas as pd
import string

df = pd.read_csv('/content/all_speeches.csv')
print('Raw speeches:', len(df))

# Drop columns we don't need
drop_cols = [c for c in ['Unnamed: 0','Year','Session','session','index',
                          'year_iso','ISO Code','Unnamed: 6','Post','Country'] if c in df.columns]
df = df.drop(columns=drop_cols)

# Rename Country to country (it was harmonize_data's job)
if 'country' not in df.columns and 'Country' in df.columns:
    df = df.rename(columns={'Country': 'country'})

# Fix known duplicate ISO / country name issues
iso_fixes = {
    'BLR': 'Belarus', 'COG': 'Democratic Republic of Congo',
    'CZK': 'Czechoslovakia', 'CSK': 'Czechoslovakia', 'RUS': 'Russia', 'UKR': 'Ukraine',
}
for iso, name in iso_fixes.items():
    df.loc[df['iso'] == iso, 'country'] = name

df = df[df['iso'] != '.DS'].reset_index(drop=True)
print('After cleanup:', len(df), 'speeches,', df['country'].nunique(), 'countries')

In [ ]:
# ── Cell 4: Split speeches into paragraphs ────────────────────────────────────
MIN_PARAGRAPH_LENGTH = 80  # chars; shorter lines are noise/headings

rows = []
for _, row in df.iterrows():
    speech = str(row.get('speeches', ''))
    # Split on single newline; each line is a paragraph
    paragraphs = [p.strip() for p in speech.split('\n')]
    paragraphs = [p for p in paragraphs if len(p) >= MIN_PARAGRAPH_LENGTH]
    for i, para in enumerate(paragraphs):
        rows.append({
            'iso':                    row['iso'],
            'year':                   row['year'],
            'country':                row.get('country', ''),
            'Name of Person Speaking': row.get('Name of Person Speaking', ''),
            'order':                  row.get('order', i),
            'speeches':               para,
            'speech_length':          len(para),
        })

para_df = pd.DataFrame(rows)
print(f'Paragraphs: {len(para_df):,} from {df["country"].nunique()} countries')

In [ ]:
# ── Cell 5: Add metadata columns ──────────────────────────────────────────────
# continent lookup by ISO-3 code
ISO_CONTINENT = {
    'AFG':'Asia','ALB':'Europe','DZA':'Africa','AND':'Europe','AGO':'Africa',
    'ATG':'North America','ARG':'South America','ARM':'Asia','AUS':'Oceania',
    'AUT':'Europe','AZE':'Asia','BHS':'North America','BHR':'Asia','BGD':'Asia',
    'BRB':'North America','BLR':'Europe','BEL':'Europe','BLZ':'North America',
    'BEN':'Africa','BTN':'Asia','BOL':'South America','BIH':'Europe','BWA':'Africa',
    'BRA':'South America','BRN':'Asia','BGR':'Europe','BFA':'Africa','BDI':'Africa',
    'CPV':'Africa','KHM':'Asia','CMR':'Africa','CAN':'North America','CAF':'Africa',
    'TCD':'Africa','CHL':'South America','CHN':'Asia','COL':'South America',
    'COM':'Africa','COD':'Africa','COG':'Africa','CRI':'North America','CIV':'Africa',
    'HRV':'Europe','CUB':'North America','CYP':'Europe','CZE':'Europe','CSK':'Europe',
    'CZK':'Europe','DNK':'Europe','DJI':'Africa','DMA':'North America','DOM':'North America',
    'ECU':'South America','EGY':'Africa','SLV':'North America','GNQ':'Africa',
    'ERI':'Africa','EST':'Europe','SWZ':'Africa','ETH':'Africa','FJI':'Oceania',
    'FIN':'Europe','FRA':'Europe','GAB':'Africa','GMB':'Africa','GEO':'Asia',
    'DEU':'Europe','GHA':'Africa','GRC':'Europe','GRD':'North America','GTM':'North America',
    'GIN':'Africa','GNB':'Africa','GUY':'South America','HTI':'North America',
    'HND':'North America','HUN':'Europe','ISL':'Europe','IND':'Asia','IDN':'Asia',
    'IRN':'Asia','IRQ':'Asia','IRL':'Europe','ISR':'Asia','ITA':'Europe','JAM':'North America',
    'JPN':'Asia','JOR':'Asia','KAZ':'Asia','KEN':'Africa','KIR':'Oceania','PRK':'Asia',
    'KOR':'Asia','KWT':'Asia','KGZ':'Asia','LAO':'Asia','LVA':'Europe','LBN':'Asia',
    'LSO':'Africa','LBR':'Africa','LBY':'Africa','LIE':'Europe','LTU':'Europe',
    'LUX':'Europe','MDG':'Africa','MWI':'Africa','MYS':'Asia','MDV':'Asia','MLI':'Africa',
    'MLT':'Europe','MHL':'Oceania','MRT':'Africa','MUS':'Africa','MEX':'North America',
    'FSM':'Oceania','MDA':'Europe','MCO':'Europe','MNG':'Asia','MNE':'Europe','MAR':'Africa',
    'MOZ':'Africa','MMR':'Asia','NAM':'Africa','NRU':'Oceania','NPL':'Asia','NLD':'Europe',
    'NZL':'Oceania','NIC':'North America','NER':'Africa','NGA':'Africa','MKD':'Europe',
    'NOR':'Europe','OMN':'Asia','PAK':'Asia','PLW':'Oceania','PAN':'North America',
    'PNG':'Oceania','PRY':'South America','PER':'South America','PHL':'Asia','POL':'Europe',
    'PRT':'Europe','QAT':'Asia','ROU':'Europe','RUS':'Europe','RWA':'Africa','KNA':'North America',
    'LCA':'North America','VCT':'North America','WSM':'Oceania','SMR':'Europe','STP':'Africa',
    'SAU':'Asia','SEN':'Africa','SRB':'Europe','SLE':'Africa','SGP':'Asia','SVK':'Europe',
    'SVN':'Europe','SLB':'Oceania','SOM':'Africa','ZAF':'Africa','SSD':'Africa','ESP':'Europe',
    'LKA':'Asia','SDN':'Africa','SUR':'South America','SWE':'Europe','CHE':'Europe',
    'SYR':'Asia','TWN':'Asia','TJK':'Asia','TZA':'Africa','THA':'Asia','TLS':'Asia',
    'TGO':'Africa','TON':'Oceania','TTO':'North America','TUN':'Africa','TUR':'Asia',
    'TKM':'Asia','TUV':'Oceania','UGA':'Africa','UKR':'Europe','ARE':'Asia','GBR':'Europe',
    'USA':'North America','URY':'South America','UZB':'Asia','VUT':'Oceania','VEN':'South America',
    'VNM':'Asia','YEM':'Asia','ZMB':'Africa','ZWE':'Africa','YUG':'Europe','SCG':'Europe',
    'DDR':'Europe','EU':'Europe','BLR':'Europe',
}

para_df['continent'] = para_df['iso'].map(ISO_CONTINENT).fillna('Unknown')

# decade and year_range
para_df['decade'] = (para_df['year'] // 10 * 10).astype(str) + 's'
def year_range(y):
    start = (y // 5) * 5
    return f'{start}-{start+4}'
para_df['year_range'] = para_df['year'].apply(year_range)

# cleaned_speeches (basic cleaning)
def basic_cleaning(text):
    if not isinstance(text, str):
        return ''
    text = text.replace('\n', ' ')
    text = ''.join(c for c in text if not c.isdigit())
    text = text.lower()
    for p in string.punctuation:
        text = text.replace(p, '')
    return text.strip()

para_df['cleaned_speeches'] = para_df['speeches'].apply(basic_cleaning)

# Placeholder NER columns (requires spacy — skip for now)
para_df['countries_mentioned'] = '[]'
para_df['countries_recoded']   = '[]'
para_df['top_5_words']         = '[]'
para_df['preprocessed_speech'] = para_df['cleaned_speeches']

print(para_df[['iso','year','country','continent','decade']].head(3))
print('Continents:', para_df['continent'].value_counts().to_dict())

In [ ]:
# ── Cell 6: Topic classification (SentenceTransformer, GPU-accelerated) ───────
import sys, gc, numpy as np, torch
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print('CUDA:', torch.cuda.is_available())

topics_df   = pd.read_csv('/content/topic_definitions.csv')
topic_names = topics_df['Name'].str.strip().tolist()
topic_texts = topics_df['Text'].str.strip().tolist()

model = SentenceTransformer('all-mpnet-base-v2')

print('Encoding topic anchors...')
anchor_embeddings = model.encode(topic_texts, show_progress_bar=False)

print(f'Encoding {len(para_df):,} paragraphs...')
paragraphs = para_df['speeches'].fillna('').tolist()

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

paragraph_embeddings = model.encode(
    paragraphs,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)

print('Computing similarities...')
similarities = cosine_similarity(paragraph_embeddings, anchor_embeddings)
best_idx   = similarities.argmax(axis=1)
best_score = similarities.max(axis=1)

para_df['topic']            = [topic_names[i] for i in best_idx]
para_df['topic_similarity'] = best_score.round(4)

print('\nTopic distribution:')
print(para_df['topic'].value_counts().to_string())
low = (best_score < 0.25).sum()
print(f'\nLow-confidence (< 0.25): {low:,} ({low/len(para_df)*100:.1f}%)')

In [ ]:
# ── Cell 7: Save speeches_paragraphs.csv ──────────────────────────────────────
OUT_COLS = [
    'iso','year','speeches','Name of Person Speaking','order','country',
    'speech_length','cleaned_speeches','preprocessed_speech',
    'topic','topic_similarity','top_5_words',
    'countries_mentioned','countries_recoded',
    'decade','year_range','continent',
]
out = para_df[[c for c in OUT_COLS if c in para_df.columns]]
out.to_csv('/content/speeches_paragraphs.csv', index=False)
print(f'Saved speeches_paragraphs.csv — {len(out):,} rows, {out["country"].nunique()} countries')

In [ ]:
# ── Cell 8: Regenerate UMAP + aggregations ────────────────────────────────────
import umap as umap_lib

# UMAP
print('Running UMAP (CPU, a few minutes)...')
reducer     = umap_lib.UMAP(n_components=2, random_state=42)
umap_coords = reducer.fit_transform(paragraph_embeddings)
umap_df     = pd.DataFrame(umap_coords, columns=['umap_1', 'umap_2'])
combined    = pd.concat(
    [para_df[['iso','year','country','continent','topic']].reset_index(drop=True), umap_df],
    axis=1
)
speeches_umap = (
    combined
    .groupby(['iso','year','country','continent','topic'], dropna=False)
    .agg(umap_1=('umap_1','mean'), umap_2=('umap_2','mean'), count=('umap_1','count'))
    .reset_index()
)
speeches_umap[['umap_1','umap_2']] = speeches_umap[['umap_1','umap_2']].round(4)
speeches_umap.to_csv('/content/speeches_umap.csv', index=False)
print(f'Saved speeches_umap.csv — {len(speeches_umap):,} rows')

# topic_labels
sys.path.insert(0, '/content')
from clean_countries import clean_country, to_drop
import ast

rows = []
for _, row in topics_df.iterrows():
    name = row['Name']
    rows.append({
        'topic_id':   int(row['Index']),
        'topic_name': name,
        'count':      int((para_df['topic'] == name).sum()),
        'top_5_words': str([k.strip() for k in str(row['Short_keywords']).split(',')][:5]),
    })
pd.DataFrame(rows).to_csv('/content/topic_labels.csv', index=False)
print(f'Saved topic_labels.csv')

In [ ]:
# ── Cell 9: Download all output files ─────────────────────────────────────────
from google.colab import files
for fname in ['speeches_paragraphs.csv', 'speeches_umap.csv', 'topic_labels.csv']:
    print(f'Downloading {fname}...')
    files.download(f'/content/{fname}')
print('\nCopy downloaded files to:')
print('  speeches_paragraphs.csv → data/')
print('  speeches_umap.csv       → streamlit/')
print('  topic_labels.csv        → raw_data/')